# 1- Kütüphaneler

In [28]:
import numpy as np
import pandas as pd

from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 2 - Veri Seti Oluşturulması

In [29]:
data = {
    "text": [
        "Otelin konumu harikaydı, her yere çok yakındı",
        "Personel çok nazik ve yardımseverdi",
        "Odalar temiz ve ferah, mükemmel bir ortam",
        "Kahvaltı çeşitleri harika ve lezzetliydi",
        "Tesisin bahçesi çok güzel ve huzur vericiydi",
        "Fiyat performans oranı çok iyiydi",
        "Çalışanlar her konuda yardımcı olmuşlardı",
        "Oda servisi çok hızlı ve profesyoneldi",
        "Spa ve havuz imkanları muazzamdı",
        "Gece hayatı yakınlarda çok zengindir",
        "Oda çok küçük ve havalı idi",
        "Temizlik personeli pek özenli değildi",
        "Manzara balkondan çok kötüydü",
        "Wifi hızı çok yavaş ve sürekli kesiliyor",
        "Restoran mutfağı beklentinin altında",
        "Otelde verilen hizmetler kötüydü",
        "Konfor ve rahatlık hiç yoktu",
        "Personel eğitimsiz ve kaba davranıyordu",
        "Otel çok güvensiz hissi verdi",
        "Kesinlikle tekrar gelmek istemediğim bir otel"
    ],
    
    "label": [
        "pozitif", 
        "pozitif", 
        "pozitif", 
        "pozitif", 
        "pozitif",
        "pozitif", 
        "pozitif", 
        "pozitif", 
        "pozitif", 
        "pozitif",
        "negatif", 
        "negatif", 
        "negatif", 
        "negatif", 
        "negatif",
        "negatif", 
        "negatif", 
        "negatif", 
        "negatif", 
        "negatif"
    ]
}

# 3 - Metin Verisinin (Text) Tokenize Edilmesi

- Tokenizer ile kelimeleri, LabelEncoder ile etiketleri sayısal temsillere dönüştüreceğiz.

- Deep Learning'de tokenizasyon iki aşamadan oluşur: kelimelere ayırma + sayısal temsil


In [30]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data["text"]) # kelimeleri öğrenir ve her kelimeye bir numara (index) atar
sequences = tokenizer.texts_to_sequences(data["text"]) # kelimelerin sayısal hale çevrilmiş listesi
word_index = tokenizer.word_index   # hangi sayının hangi kelime olduğunu görebilmek için

print(f"Vocab Size (Veri Setindeki Toplam Kelime Dağarcığımız): {len(word_index)}")

Vocab Size (Veri Setindeki Toplam Kelime Dağarcığımız): 84


# 4 - Tüm Veriyi Eşit Boyuta Getirme (Padding)

In [31]:
sequences

[[9, 10, 11, 3, 12, 1, 13],
 [4, 1, 14, 2, 15],
 [16, 17, 2, 18, 19, 5, 20],
 [21, 22, 23, 2, 24],
 [25, 26, 1, 27, 2, 28, 29],
 [30, 31, 32, 1, 33],
 [34, 3, 35, 36, 37],
 [6, 38, 1, 39, 2, 40],
 [41, 2, 42, 43, 44],
 [45, 46, 47, 1, 48],
 [6, 1, 49, 2, 50, 51],
 [52, 53, 54, 55, 56],
 [57, 58, 1, 7],
 [59, 60, 1, 61, 2, 62, 63],
 [64, 65, 66, 67],
 [68, 69, 70, 7],
 [71, 2, 72, 73, 74],
 [4, 75, 2, 76, 77],
 [8, 1, 78, 79, 80],
 [81, 82, 83, 84, 5, 8]]

- Makine öğrenmesi modelleri, sabit boyutta giriş ister ama metinler farklı uzunlukta.
- Aşağıda yapacağımız padding işlemi ile birlikte farklı uzunluktaki metin dizilerini, modelin anlayabileceği şekilde aynı uzunluğa getireceğiz.
- Bunun için ilk olarak en uzun cümleyi buluyoruz ki hepsini buna eşitleyeceğiz. Kısa olanların başına "0" ekleyeceğiz.

In [32]:
# sequences: İçinde sayısal hale getirilmiş metin dizileri bulunan liste
# Örn: [[1, 5, 8], [2, 3], [4, 7, 9, 10]]

# Her bir dizinin uzunluğunu hesapla ve en uzun olanı bul
maxlen = max(len(seq) for seq in sequences)

# Tüm dizileri aynı uzunluğa getirmek için padding uygula
# Kısa olan dizilerin başına 0 eklenir (varsayılan: pre-padding)
x = pad_sequences(sequences, maxlen=maxlen)

# Oluşan yeni dizinin boyutunu yazdır
# (satır sayısı, her dizinin uzunluğu)
print(f"Shape of x (Padded Sequences): {x.shape}")

Shape of x (Padded Sequences): (20, 7)


In [33]:
sequences

# Metinler tokenize edilir (kelimelere ayrılır)
# Her kelimeye bir numara atanır

# Her sayı → bir kelimeyi temsil ediyor
# Aynı sayı → aynı kelime

[[9, 10, 11, 3, 12, 1, 13],
 [4, 1, 14, 2, 15],
 [16, 17, 2, 18, 19, 5, 20],
 [21, 22, 23, 2, 24],
 [25, 26, 1, 27, 2, 28, 29],
 [30, 31, 32, 1, 33],
 [34, 3, 35, 36, 37],
 [6, 38, 1, 39, 2, 40],
 [41, 2, 42, 43, 44],
 [45, 46, 47, 1, 48],
 [6, 1, 49, 2, 50, 51],
 [52, 53, 54, 55, 56],
 [57, 58, 1, 7],
 [59, 60, 1, 61, 2, 62, 63],
 [64, 65, 66, 67],
 [68, 69, 70, 7],
 [71, 2, 72, 73, 74],
 [4, 75, 2, 76, 77],
 [8, 1, 78, 79, 80],
 [81, 82, 83, 84, 5, 8]]

In [34]:
x # hepsi aynı uzunlukta oldu, kısa olanların başına 0 eklendi

array([[ 9, 10, 11,  3, 12,  1, 13],
       [ 0,  0,  4,  1, 14,  2, 15],
       [16, 17,  2, 18, 19,  5, 20],
       [ 0,  0, 21, 22, 23,  2, 24],
       [25, 26,  1, 27,  2, 28, 29],
       [ 0,  0, 30, 31, 32,  1, 33],
       [ 0,  0, 34,  3, 35, 36, 37],
       [ 0,  6, 38,  1, 39,  2, 40],
       [ 0,  0, 41,  2, 42, 43, 44],
       [ 0,  0, 45, 46, 47,  1, 48],
       [ 0,  6,  1, 49,  2, 50, 51],
       [ 0,  0, 52, 53, 54, 55, 56],
       [ 0,  0,  0, 57, 58,  1,  7],
       [59, 60,  1, 61,  2, 62, 63],
       [ 0,  0,  0, 64, 65, 66, 67],
       [ 0,  0,  0, 68, 69, 70,  7],
       [ 0,  0, 71,  2, 72, 73, 74],
       [ 0,  0,  4, 75,  2, 76, 77],
       [ 0,  0,  8,  1, 78, 79, 80],
       [ 0, 81, 82, 83, 84,  5,  8]], dtype=int32)

# 5 - Label Encoding (Etiketlerin Sayısal Temsillere Dönüştürülmesi)

In [35]:
data["label"]

['pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'pozitif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif']

In [36]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data["label"])
print(f"Encoded Labels: {y}") 

# pozitif → 1
# negatif → 0 atadı

Encoded Labels: [1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0]


# 6- Train-Test Split

In [37]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

# 7 - Word Embedding: Word2Vec, Embedding Matrisi Oluşturma

In [38]:
sentences = [text.split()for text in data["text"]]
sentences

[['Otelin', 'konumu', 'harikaydı,', 'her', 'yere', 'çok', 'yakındı'],
 ['Personel', 'çok', 'nazik', 've', 'yardımseverdi'],
 ['Odalar', 'temiz', 've', 'ferah,', 'mükemmel', 'bir', 'ortam'],
 ['Kahvaltı', 'çeşitleri', 'harika', 've', 'lezzetliydi'],
 ['Tesisin', 'bahçesi', 'çok', 'güzel', 've', 'huzur', 'vericiydi'],
 ['Fiyat', 'performans', 'oranı', 'çok', 'iyiydi'],
 ['Çalışanlar', 'her', 'konuda', 'yardımcı', 'olmuşlardı'],
 ['Oda', 'servisi', 'çok', 'hızlı', 've', 'profesyoneldi'],
 ['Spa', 've', 'havuz', 'imkanları', 'muazzamdı'],
 ['Gece', 'hayatı', 'yakınlarda', 'çok', 'zengindir'],
 ['Oda', 'çok', 'küçük', 've', 'havalı', 'idi'],
 ['Temizlik', 'personeli', 'pek', 'özenli', 'değildi'],
 ['Manzara', 'balkondan', 'çok', 'kötüydü'],
 ['Wifi', 'hızı', 'çok', 'yavaş', 've', 'sürekli', 'kesiliyor'],
 ['Restoran', 'mutfağı', 'beklentinin', 'altında'],
 ['Otelde', 'verilen', 'hizmetler', 'kötüydü'],
 ['Konfor', 've', 'rahatlık', 'hiç', 'yoktu'],
 ['Personel', 'eğitimsiz', 've', 'kaba', '

In [39]:
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# vector_size: Kelimelerin vektör boyutu (embedding boyutu)
# window: Kelimenin çevresindeki kelimeleri dikkate alma penceresi (örneğin window=5 ise, her kelimenin çevresindeki 5 kelime dikkate alınır)
# min_count: Kelimenin kaç kere geçmesi gerektiği (1 ise tüm kelimeler dahil edilir)
# workers: Eğitim sırasında kullanılacak iş parçacığı sayısı (CPU çekirdeği sayısına göre ayarlanabilir)

In [40]:
embdeding_dim = 100
embedding_matrix = np.zeros((len(word_index)+1, embdeding_dim))
embedding_matrix


# embedding_dim: Kelime vektörlerinin boyutu (Word2Vec modelinde belirttiğimiz vector_size ile aynı olmalı)
# embedding_matrix: Kelime indekslerine karşılık gelen vektörleri içeren matris

# Sıfırlarla dolu bir matris oluşturduk, şimdi kelimelerin vektörlerini bu matrise yerleştireceğiz (bir alt satırda)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(85, 100))

In [41]:
# Kelimelerin vektörlerini oluşuturduğumuz embedding_matrix'e yerleştirelim

for word, i in word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

embedding_matrix    # kelimlerin vektörlerini üstte oluşturduğumuz matrise yerleştirdik, şimdi bu matrisi embedding katmanında kullanacağız

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.00863361,  0.0036798 ,  0.00518778, ..., -0.00239424,
        -0.00951126,  0.00449333],
       [-0.00053845,  0.0002394 ,  0.0051023 , ..., -0.00703939,
         0.00089431,  0.00638652],
       ...,
       [ 0.00707261, -0.00156289,  0.00795144, ..., -0.00546654,
         0.00380078, -0.00813254],
       [-0.00515738, -0.00666822, -0.00777484, ...,  0.00584278,
         0.00939463,  0.00349537],
       [-0.00959945,  0.00894441,  0.00416261, ...,  0.00713322,
         0.00586648, -0.00560125]], shape=(85, 100))

# 8 - RNN Modelinin Oluşturulması

In [67]:
rnn_model = Sequential()
rnn_model.add(Embedding(input_dim=len(word_index)+1, output_dim=embdeding_dim, weights=[embedding_matrix], input_length=maxlen, trainable=False)) #öğrenme gerçekleşmeyeceği için trainable=False yaptık, embedding_matrix'i kullanarak kelimelerin vektörlerini öğrenmeyeceğiz, zaten word2vec ile öğrendik, sadece kullanacağız
rnn_model.add(SimpleRNN(64, activation="relu", return_sequences=False)) # return_sequences=False yaptık çünkü sadece son çıktıyı almak istiyoruz, eğer True olsaydı her zaman bir çıktı verecekti, biz sadece son çıktıyı almak istediğimiz için False yapıyoruz
rnn_model.add(Dense(1, activation="sigmoid")) # sigmoid aktivasyon fonksiyonu kullandık çünkü ikili sınıflandırma yapacağız, çıktı 0 ile 1 arasında olacak, 0 negatif, 1 pozitif anlamına gelecek


c:\Users\Bedirhan Orseloglu\OneDrive\Masaüstü\Turkcell Eğitim\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [68]:
rnn_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# 9 - Modelin Eğitimi

In [69]:
rnn_model.fit(x_train, y_train, epochs=10, batch_size=2, validation_data=(x_test, y_test))

Epoch 1/10


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.4286 - loss: 0.6929 - val_accuracy: 0.5000 - val_loss: 0.6929
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8571 - loss: 0.6887 - val_accuracy: 0.5000 - val_loss: 0.6930
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9286 - loss: 0.6862 - val_accuracy: 0.3333 - val_loss: 0.6942
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7857 - loss: 0.6818 - val_accuracy: 0.3333 - val_loss: 0.6943
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7857 - loss: 0.6777 - val_accuracy: 0.3333 - val_loss: 0.6944
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7857 - loss: 0.6714 - val_accuracy: 0.3333 - val_loss: 0.6922
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8571 - loss: 0.6638 - val_accuracy: 0.3333 - val_loss: 0.6919
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7143 - loss: 0.6560 - val_accuracy: 0.3333 - val_loss: 0.6936
Epoch 9/10
7/7 ━━━

# 10 - Modelin Değerlendirilmesi

In [70]:
loss, accuracy = rnn_model.evaluate(x_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Veri setimiz çok küçük olduğu için modelimizin performansı düşük olabilir, daha büyük bir veri seti ile daha iyi sonuçlar elde edebiliriz

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.3333 - loss: 0.6894
Test Loss: 0.6893600821495056
Test Accuracy: 0.3333333432674408


# PS: Cümle Sınıflandırıcı

In [74]:
def cumle_siniflandirici(cumle):
    cumle = tokenizer.texts_to_sequences([cumle]) # cümleyi tokenize eder ve sayısal hale getirir
    padded_cumle = pad_sequences(cumle, maxlen=maxlen) # cümleyi padding yaparak aynı uzunluğa getirir

    prediction = rnn_model.predict(padded_cumle) # cümlenin hangi sınıfa ait olduğunu tahmin eder, çıktı 0 ile 1 arasında olacak, 0 negatif, 1 pozitif anlamına gelecek
    predicted_class = (prediction > 0.5).astype(int) # tahmin edilen değeri 0 veya 1'e dönüştürür, 0.5'ten büyükse 1 (pozitif), küçükse 0 (negatif) olarak sınıflandırır
    label = "pozitif" if predicted_class[0][0] == 1 else "negatif" 
    return label

In [75]:
cumle_siniflandirici("Otelin hizmet kalitesini hiç beğenmedim") # negatif

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step


'negatif'

In [76]:
cumle_siniflandirici("Otelin konumu harikaydı, her yere çok yakındı") # pozitif

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


'negatif'